# Movie Recommendation System

**Content-based filtering** using TF-IDF + NearestNeighbors (cosine similarity)

| Step | What happens |
|------|--------------|
| 1 | Load `movies.csv` and `ratings.csv` |
| 2 | Preprocess (year, tags, rating stats) |
| 3 | Filter movies with ≥ 500 ratings |
| 4 | Build TF-IDF matrix from genre tags |
| 5 | Train NearestNeighbors model |
| 6 | Get top-10 similar movies for any title |

## Section 1 — Imports

In [ ]:
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors


## Section 2 — Load Data

In [ ]:
DATA_DIR = r"C:/Users/abinj/OneDrive/Desktop/project/ML/moive bar"

movies  = pd.read_csv(os.path.join(DATA_DIR, "movies.csv"),  usecols=["movieId", "title", "genres"])
ratings = pd.read_csv(os.path.join(DATA_DIR, "ratings.csv"), usecols=["movieId", "rating"])

print(f"movies  : {len(movies):,} rows")
print(f"ratings : {len(ratings):,} rows")


## Section 3 — Preprocess

In [ ]:
# --- Fill missing genres ---
movies["genres"] = movies["genres"].fillna("")

# --- Extract year from title, e.g. 'Toy Story (1995)' -> '1995' ---
movies["year"] = movies["title"].str.extract(r"\((\d{4})\)")

# --- Genre string used as TF-IDF input ---
movies["tags"] = movies["genres"]

# --- Aggregate rating stats per movie ---
stats = (
    ratings
    .groupby("movieId", as_index=False)
    .agg(avg_rating=("rating", "mean"), rating_count=("rating", "count"))
)
stats["avg_rating"] = stats["avg_rating"].round(2)

# --- Merge into movies ---
movies = movies.merge(stats, on="movieId", how="left")
movies["avg_rating"]   = movies["avg_rating"].fillna(0.0)
movies["rating_count"] = movies["rating_count"].fillna(0).astype(int)

print("Preprocessing complete.")
movies[["title", "year", "avg_rating", "rating_count"]].head()


## Section 4 — Filter (rating_count ≥ 500)

In [ ]:
# Keep only movies with at least 500 user ratings
MIN_RATINGS = 500
movies = movies[movies["rating_count"] >= MIN_RATINGS].copy()

# Reset index so it stays contiguous and aligned with TF-IDF matrix
movies.reset_index(drop=True, inplace=True)
print(f"Movies after filter (>= {MIN_RATINGS} ratings): {len(movies):,}")


## Section 5 — TF-IDF Feature Matrix

In [ ]:
# Token pattern '[^|]+' treats each genre as a single token (splits on '|')
tfidf        = TfidfVectorizer(token_pattern=r"[^|]+")
tfidf_matrix = tfidf.fit_transform(movies["tags"])

print(f"TF-IDF matrix: {tfidf_matrix.shape}")
print(f"  rows = movies in filtered df = {len(movies):,}  (must match)")


## Section 6 — Train NearestNeighbors Model

In [ ]:
# n_neighbors=11: first result is the query movie itself, so we get top 10 others
model = NearestNeighbors(n_neighbors=11, metric="cosine", algorithm="brute", n_jobs=-1)
model.fit(tfidf_matrix)
print("NearestNeighbors model trained.")


## Section 7 — Recommendation Function

In [ ]:
def recommend_movies(title: str, top_n: int = 10) -> None:
    mask    = movies["title"].str.contains(title, case=False, regex=False)
    matches = movies[mask]
    if matches.empty:
        print(f"[Not Found] No movie matching '{title}' in the dataset.")
        return

    idx         = matches.index[0]
    found_title = movies.loc[idx, "title"]
    vec         = tfidf_matrix[idx]
    distances, indices = model.kneighbors(vec, n_neighbors=top_n + 1)
    distances = distances[0][1:]
    indices   = indices[0][1:]

    sep = "=" * 68
    print(f"\n{sep}")
    print(f"  Top {top_n} recommendations for: {found_title}")
    print(sep)

    for rank, (i, dist) in enumerate(zip(indices, distances), start=1):
        rec_title  = movies.loc[i, "title"]
        avg_rating = movies.loc[i, "avg_rating"]
        similarity = (1 - dist) * 100
        print(f"  {rank:>2}. {rec_title:<44}| Similarity: {similarity:5.2f}%| Rating: {avg_rating:.2f}")
    print(f"{sep}\n")
\n

## Section 8 — Run Recommendations

Change the title string to get recommendations for any movie.

In [ ]:
recommend_movies("Toy Story")

In [ ]:
recommend_movies("Iron Man")

In [ ]:
recommend_movies("The Dark Knight")

In [ ]:
recommend_movies("Inception")